# Day 3 — Airflow DAG Scheduling

Production RAG pipelines need scheduled re-indexing. Apache Airflow is the industry standard for orchestrating data pipelines.
Key concepts: DAG, PythonOperator, cron schedule, idempotency, task dependencies.
Teams at Shell, Walmart, and Landmark Group use Airflow (or its cloud equivalents) to run daily re-indexing jobs.

In [ ]:
import sys
sys.path.insert(0, '../src')
from datetime import datetime
from day3.airflow_demo import (
    extract_products, chunk_and_embed,
    index_to_vectorstore, validate_index, notify_complete,
    SimpleDAGRunner, create_rag_ingestion_dag_definition,
    show_airflow_code, try_real_airflow, TaskResult
)
print("Airflow demo module loaded")
print("Note: Airflow is not required to be installed for this notebook")

## 1. DAG Structure

A DAG (Directed Acyclic Graph) defines the pipeline structure.
Tasks have explicit dependencies: extract \u2192 chunk \u2192 index \u2192 validate \u2192 notify. No cycles allowed.
Airflow executes tasks in topological order.

In [ ]:
dag_def = create_rag_ingestion_dag_definition()
print(f"DAG ID:    {dag_def['dag_id']}")
print(f"Schedule:  {dag_def['schedule_interval']}  (cron: {dag_def['cron_expression']})")
print(f"Timezone:  {dag_def['timezone']}")
print(f"Start:     {dag_def['start_date']}")
print(f"\nTask graph:")
for task in dag_def["tasks"]:
    deps = " \u2192 ".join(task["depends_on"]) if task["depends_on"] else "(start)"
    retries = task["retry_policy"]["retries"]
    print(f"  [{task['task_id']:<10}] after {deps:<15} | {task['description']}")
    print(f"              operator={task['operator']}, retries={retries}")

## 2. Idempotency Pattern

The most important concept in production pipelines.
Running a task twice gives the same result as running once.
Each task checks if its output file already exists before re-processing \u2014 safe to re-run after failures.

In [ ]:
execution_date = "2026-01-15"

print(f"First run \u2014 execution_date={execution_date}")
result1 = extract_products(execution_date=execution_date)
print(f"  Extracted: {result1['count']} records")
print(f"  Source: {result1['source_path']}")

print(f"\nSecond run \u2014 same execution_date (should be IDEMPOTENT):")
result2 = extract_products(execution_date=execution_date)
print(f"  Extracted: {result2['count']} records")
print(f"  This was read from /tmp/dag_outputs/{execution_date}/extract.json \u2014 not re-processed")

assert result1["count"] == result2["count"], "Idempotency violated!"
print("\nIdempotency confirmed \u2713")

## 3. Running the Full Pipeline with SimpleDAGRunner

SimpleDAGRunner simulates Airflow's execution model: topological sort, task dependency passing, result tracking.
Same logic as Airflow \u2014 just without the web UI and scheduler.

In [ ]:
runner = SimpleDAGRunner("rag_product_ingestion")

runner.add_task("extract",  extract_products,    depends_on=[])
runner.add_task("chunk",    chunk_and_embed,      depends_on=["extract"])
runner.add_task("index",    index_to_vectorstore, depends_on=["chunk"])
runner.add_task("validate", validate_index,       depends_on=["index"])
runner.add_task("notify",   notify_complete,      depends_on=["validate"])

execution_date_2 = "2026-01-16"
results = runner.run(execution_date_2)

## 4. Task Results

Each task returns a TaskResult with status, output, and timing.
In production Airflow, these appear in the web UI as green/red task boxes.

In [ ]:
print(f"{'Task':<12} {'Status':<10} {'Duration':>10}")
print("-" * 36)
total = 0
for r in results:
    icon = "\u2713" if r.status == "success" else "\u2717"
    print(f"{icon} {r.task_id:<11} {r.status:<10} {r.duration_s:>9.3f}s")
    total += r.duration_s
print(f"\nTotal pipeline time: {total:.3f}s")

# Show what each task produced
print("\nTask outputs:")
for r in results:
    if isinstance(r.output, dict):
        output_keys = list(r.output.keys())
        print(f"  {r.task_id}: {output_keys}")
    else:
        print(f"  {r.task_id}: {str(r.output)[:80]}")

## 5. Real Airflow Code

This is the actual Python DAG file you would deploy to an Airflow cluster.
The extract >> chunk >> index >> validate >> notify syntax is the Airflow bit-shift dependency operator.

In [ ]:
airflow_result = try_real_airflow()
if isinstance(airflow_result, dict) and airflow_result.get("status") == "airflow_installed":
    print(f"Airflow installed! DAG created: {airflow_result['dag_id']}")
else:
    print("Airflow not installed \u2014 here is the real DAG code:")
    print(show_airflow_code())

## 6. Cron Schedule Reference

'0 2 * * *' means 2:00 AM every day.
Cron is the standard scheduling language for Airflow, Linux crontabs, and most cloud schedulers.

In [ ]:
schedules = [
    ("0 2 * * *",      "2 AM daily"),
    ("0 */6 * * *",    "Every 6 hours"),
    ("0 2 * * 1",      "2 AM every Monday"),
    ("0 2 1 * *",      "2 AM on the 1st of every month"),
    ("*/15 * * * *",   "Every 15 minutes"),
    ("@daily",         "Once per day (midnight)"),
    ("@hourly",        "Once per hour"),
]

print("Cron Expression Reference:")
print(f"{'Cron':<20} {'Meaning'}")
print("-" * 45)
for cron, meaning in schedules:
    print(f"  {cron:<18} {meaning}")

print("""
Cron format:
  \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500 minute    (0\u201359)
  \u2502 \u250c\u2500\u2500\u2500\u2500\u2500 hour      (0\u201323)
  \u2502 \u2502 \u250c\u2500\u2500\u2500 day       (1\u201331)
  \u2502 \u2502 \u2502 \u250c\u2500 month     (1\u201312)
  \u2502 \u2502 \u2502 \u2502 \u250c day-of-week (0\u20137, 0=Sunday)
  \u2502 \u2502 \u2502 \u2502 \u2502
  * * * * *
""")

## 7. Databricks Workflows Bridge

Databricks Workflows is the cloud equivalent of Airflow \u2014 managed, no infrastructure to maintain, native Delta Lake integration.

In [ ]:
print("""
DATABRICKS WORKFLOWS EQUIVALENT:

  Databricks UI \u2192 Workflows \u2192 Create Job \u2192 Add Tasks
  
  Task 1: extract_products
    - Type: Python script / Notebook
    - Cluster: Shared (autoscaling)
    - Parameters: execution_date={{date}}
  
  Task 2: chunk_and_embed  
    - Depends on: extract_products
    - Type: Python script
  
  Task 3: index_to_vectorstore
    - Depends on: chunk_and_embed
    - Databricks Vector Search sync (built-in, no manual code)
  
  Task 4: validate_index
    - Depends on: index_to_vectorstore
    - Run notebook with test queries
  
  Schedule: 0 2 * * *  (2 AM IST daily)
  Alerts:   Email on failure \u2192 naval@datamasterconsulting.com
  
  vs Apache Airflow:
    Airflow: you manage the server, scheduler, workers
    Databricks Workflows: fully managed, auto-scales, native Unity Catalog
""")